# GNN Cloud Notebook: GCN-3 Ensemble Residual Analysis

This standalone notebook analyzes the completed GCN-3 ensemble multi-split run without retraining any models. It uses the saved member predictions and existing lattice CSV files to locate the remaining error patterns.

Analysis outputs include:
- residual metrics for every test split and prediction-set run
- error by target-stiffness quantile
- residual correlations with graph size, density, edge weights, degree, and geometric spread
- train/test/prediction feature-distribution shift
- ensemble disagreement versus actual error
- sample stability and ranked worst-case lattices
- CSV, JSON, and PNG artifacts suitable for GitHub


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')
print('No GPU or model training dependencies are required for this analysis.')


In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')


In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
USE_DRIVE_FOR_ENSEMBLE_RESULTS = False
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_ENSEMBLE_RUN_DIR = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_multi_split/run_resumable_v1'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/gcn3_ensemble_residual_analysis'
ENSEMBLE_RUN_NAME = 'run_resumable_v1'
PUSH_RESULTS_TO_GITHUB = False
GITHUB_TOKEN = ''
GIT_BRANCH = 'main'
GIT_COMMIT_USERNAME = ''
GIT_COMMIT_EMAIL = ''
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/gcn3_ensemble_residual_analysis'
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

repo_root = Path(REPO_DIR).resolve() if IN_COLAB else Path.cwd().resolve()
pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
gnn_root = pipeline_root / 'gnn_prototype'
optimization_dir = gnn_root / 'GCN_Optimization'
if not (optimization_dir / 'residual_error_analysis.py').is_file():
    raise FileNotFoundError(f'Residual analysis runner not found at {optimization_dir}')

os.chdir(pipeline_root)
if str(optimization_dir) not in sys.path:
    sys.path.insert(0, str(optimization_dir))

use_drive = USE_DRIVE_FOR_DATA or USE_DRIVE_FOR_ENSEMBLE_RESULTS or SAVE_OUTPUTS_TO_DRIVE
if IN_COLAB and use_drive:
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if USE_DRIVE_FOR_ENSEMBLE_RESULTS:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_ENSEMBLE_RESULTS is only supported in Colab.')
    ensemble_run_dir = Path(DRIVE_ENSEMBLE_RUN_DIR)
else:
    ensemble_run_dir = gnn_root / 'outputs' / 'gcn3_ensemble_multi_split' / ENSEMBLE_RUN_NAME

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/gcn3_ensemble_residual_analysis') if IN_COLAB else gnn_root / 'outputs' / 'gcn3_ensemble_residual_analysis'

if not (ensemble_run_dir / 'gcn3_ensemble_multi_split_summary.csv').is_file():
    raise FileNotFoundError(
        f'Completed ensemble results not found at {ensemble_run_dir}. '
        'Set USE_DRIVE_FOR_ENSEMBLE_RESULTS=True if the run is only in Google Drive.'
    )

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_root.mkdir(parents=True, exist_ok=True)
if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email: ').strip()

print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Ensemble results: {ensemble_run_dir}')
print(f'Analysis output root: {output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')


In [ ]:
import json
import pandas as pd
import shutil
import subprocess
from IPython.display import Image, display

from residual_error_analysis import ResidualAnalysisConfig, run_residual_error_analysis


In [ ]:
SPLIT_SEEDS = (11, 42, 73, 101, 202)
STIFFNESS_BINS = 5
FEATURE_BINS = 5
UNCERTAINTY_BINS = 5
WORST_CASE_COUNT = 25

config = ResidualAnalysisConfig(
    split_seeds=SPLIT_SEEDS,
    stiffness_bins=STIFFNESS_BINS,
    feature_bins=FEATURE_BINS,
    uncertainty_bins=UNCERTAINTY_BINS,
    worst_case_count=WORST_CASE_COUNT,
)
analysis_run_dir = output_root / f'run_{RUN_STAMP}'
print(config)
print(f'Analysis run directory: {analysis_run_dir}')


In [ ]:
result = run_residual_error_analysis(
    config,
    train_root=train_root,
    predict_root=predict_root,
    ensemble_run_dir=ensemble_run_dir,
    run_dir=analysis_run_dir,
)

output_dir = result['output_dir']
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

print('Mean residual metrics by split:')
display(result['split_metrics'])
print('Error by stiffness bin:')
display(result['stiffness_metrics'])
print('Ensemble uncertainty correlations:')
display(result['uncertainty_correlations'])
print('Largest distribution shifts:')
display(result['shift_summary'].groupby('Comparison', group_keys=False).head(8))
print('Worst residual cases:')
display(result['worst_cases'])


In [ ]:
figure_names = [
    'residual_overview.png',
    'stiffness_bin_errors.png',
    'feature_error_correlations.png',
    'feature_distribution_shift.png',
    'ensemble_uncertainty_analysis.png',
]
for figure_name in figure_names:
    print(figure_name)
    display(Image(filename=str(output_dir / figure_name)))

print('Saved analysis files:')
for path in sorted(output_dir.iterdir()):
    if path.is_file():
        print(f' - {path.name}')


In [ ]:
if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / output_dir.name
    git_output_root.mkdir(parents=True, exist_ok=True)
    if git_run_dir.exists():
        shutil.rmtree(git_run_dir)
    shutil.copytree(output_dir, git_run_dir)
    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)
    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)
    try:
        subprocess.run(
            ['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')],
            check=True,
        )
        diff_result = subprocess.run(['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'], check=False)
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add GCN-3 ensemble residual analysis for {output_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed residual analysis to {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)
else:
    print('GitHub push disabled.')


In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_base = f'/content/{output_dir.name}_residual_analysis'
    archive_path = shutil.make_archive(archive_base, 'zip', root_dir=output_dir)
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')
